# Spotify Genre Classification

In [ ]:
cd '/Users/wiseer/Documents/github/listen-wiseer/src/'

In [ ]:
from analysis.data import *
from analysis.genre import *
from analysis.plotting import *
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

from models.clustering import *
from utils.const import *

data = LoadPlaylistData()

In [ ]:
df = data.return_enoa_data()
# time_signature, key_mode, decade,
# 'energy', 'loudness','speechiness', 'acousticness', 'instrumentalness', 'liveness','valence', 'tempo','popularity, 'top', 'left'

In [ ]:
plot_genres_dendrogram(df)

In [ ]:
n_components = select_pca_n_components(X)

In [ ]:
n_clusters = plot_elbow_method(X)

In [ ]:
plot_sihloette_scores(X)

In [ ]:
def config_model_pipeline(n_components, n_clusters):
    pipelines = []
    pipelines.append(
        Pipeline(
            [
                (
                    "scaler",
                    MinMaxScaler(),
                ),
                ("pca", PCA(n_components=n_components)),
                ("classifier", KMeans(n_clusters=n_clusters, n_init=10)),
            ]
        )
    )
    pipelines.append(
        Pipeline(
            [
                (
                    "scaler",
                    MinMaxScaler(),
                ),
                ("pca", PCA(n_components=n_components)),
                ("classifier", SpectralClustering(n_clusters=n_clusters)),
            ]
        )
    )
    # pipelines.append(
    #    Pipeline([
    #        ("scaler", MinMaxScaler(),),
    #        ("pca", PCA(n_components=n_components)),
    #        ("classifier", DBSCAN(eps=1, min_samples=2)),
    #    ])
    # )
    # pipelines.append(
    #    Pipeline([
    #        ("scaler", MinMaxScaler(),),
    #        ("pca", PCA(n_components=n_components)),
    #        ("classifier", OPTICS(min_samples=2)),
    #    ])
    # )
    return pipelines

In [ ]:
pipelines = config_model_pipeline(n_components, n_clusters)

# return clusters
for model in pipelines:
    model_name = type(model.named_steps["classifier"]).__name__
    labels = model.fit_predict(X).tolist()
    df[model_name] = labels

    # return metrics for models
    silhouette_avg = silhouette_score(X, labels)
    print(model_name + f" Silhouette Score: {silhouette_avg}")
    ch_score = calinski_harabasz_score(X, labels)
    print(model_name + f" Calinski-Harabasz Index: {ch_score}")

    # compare cluster features
    plot_tsne(X, n_components, labels, model_name)
    plot_cluster_features(df, model_name)

In [ ]:
# df[df.KMeans == 0]['genres'].value_counts().plot(kind='bar')
df.KMeans.value_counts().plot(kind="bar")

In [ ]:
df[df.KMeans == 0][["track_name", "artist_names", "genres"]]

In [ ]:
df.SpectralClustering.value_counts().plot(kind="bar")

In [ ]:
df[df.SpectralClustering == 0][["track_name", "artist_names", "genres"]]

In [ ]:
X